# Getting Started with Ollama Langchain
This link discusses downloading ollama, pulling a model, then running it on windows or linux

https://python.langchain.com/docs/integrations/chat/ollama/

In [3]:
!pip install -qU langchain-ollama

In [3]:
!pip install langgraph>0.2.27

## Initialize the Llama Model

In [1]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model='llama3.2',
    temperature=0 # ranges from 0 to 1
)

API Reference for initializing the Llama Model
https://python.langchain.com/api_reference/ollama/chat_models/langchain_ollama.chat_models.ChatOllama.html

## Simple Invocation - Prompt and Get Content

In [2]:
from langchain_core.messages import AIMessage

messages = [
    (
        'system',
        'You are a helpful assistant that translates English to Spanish. Translate the inputted sentence.',
    ),
    (
        'human',
        'Goat cheese is delicious.'
    )
]

ai_msg = llm.invoke(messages)
ai_msg

AIMessage(content='El queso de cabra es delicioso.', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2025-01-12T19:38:50.7579035Z', 'message': {'role': 'assistant', 'content': ''}, 'done_reason': 'stop', 'done': True, 'total_duration': 3990048200, 'load_duration': 3084752700, 'prompt_eval_count': 48, 'prompt_eval_duration': 266000000, 'eval_count': 12, 'eval_duration': 200000000}, id='run-077ad32c-8102-47c5-94ce-6f842ae1eba5-0', usage_metadata={'input_tokens': 48, 'output_tokens': 12, 'total_tokens': 60})

In [3]:
print(ai_msg.content)

El queso de cabra es delicioso.


# Chaining with a prompt template

In [4]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant that translates {input_language} to {output_language}. Translate the inputted sentence.",
        ),
        (
            'human',
            '{input}',
        )
    ]
)
chain = prompt | llm
ai_msg = chain.invoke(
            {
                'input_language': 'English',
                'output_language': 'French',
                'input': 'The goat cheese is delicious.'
            }
        )

In [5]:
ai_msg.content

'Le fromage de chèvre est délicieux.\n\n(Note: "goat cheese" can also be translated as "fromage de chèvre", which is a more common and idiomatic expression in French.)'

In [6]:
ai_msg.content

'Le fromage de chèvre est délicieux.\n\n(Note: "goat cheese" can also be translated as "fromage de chèvre", which is a more common and idiomatic expression in French.)'

# Testing Ollama to print and run python code for simple addition

Langchain Tool:
Interfaces that an agent, chain, or LLM can use to interact with the world. 
Provides them a function to be able to call (an action to tack)

In [8]:
import os
import traceback
from io import StringIO
from contextlib import redirect_stdout

import langchain
from langchain_ollama import ChatOllama
from langchain.chains import LLMChain
from langchain_core.prompts import PromptTemplate
from langchain.tools import Tool

# Initialize LLM
llm = llm = ChatOllama(
    model='llama3.2',
    temperature=0 # ranges from 0 to 1
)

# Define the Python execution tool (Langchain tool)
def execute_python_code(code:str)->str:
    '''
    Executes the python code provided in a string. Captures output and errors and to return them
    '''
    try:
        # Capture standard output
        f = StringIO()
        with redirect_stdout(f):
            exec(code,{})

        # Return captured output
        return f.getvalue() if f.getvalue() else "Code executed without output."
    except:
        return f'Execution error: {traceback.format_exc()}'

# Define the langchain tool for executing python code
python_executor = Tool(name='PythonExecutor',
                      func=execute_python_code,
                      description='Executes Python code and returns the output or error.'
                      )
# Create the prompt 
prompt_template = """
You are an AI assistant with a Python interpreter. When quantitative analysis or code execution is required for assisting your reasoning, you may generate Python code. 
Generated code should always include necessary imports and provide output relevant to the task.
Start python code using ```python and end it using ```.
If there’s an error during execution, I will return the error message for you to adjust the code and try again.
If you are satisfied with your answer after you have received the code execution results, output "DONE: ANSWER" and provide the answer from the code execution.

User Prompt: {user_prompt}
"""

prompt=PromptTemplate(input_variables=['user_prompt'], template=prompt_template)

# Connect the chain
llm_chain = prompt | llm

Main Dialogue Loop

In [42]:
def interactive_code_execution():
    namespace = {}
    ct = 0
    while ct<2:
        user_input = input('You: ')

        # Exit if user types 'exit'
        if user_input.lower() == 'exit':
            print('Exiting Dialogue.')
            break
        
        # Generate the LLM response with potential code to execute
        response = llm_chain.invoke({'user_prompt': user_input})
        print('\nLLM Response: ',response.content, '\n')

        # Check for Python code blocks in the LLM response
        if "```python" in response.content:
            code_snippet = response.content.split("```python")[1].split("```")[0].strip()
            print('\nExecuting code snippet:\n', code_snippet)
            try:
                # Execute the code
                execution_result = python_executor.run(code_snippet)
    
                # Append execution results to continue the conversation
                print('\nExecution Result:', execution_result)
                result_context = (f"User asked: '{user_input}'\n"
                        f"Your response: '{response.content}'\n"
                        f"Your Code Execution Result: {execution_result}\n"   
                )
                
                # Pass the result back as context to refine or complete the response
                response = llm_chain.invoke({'user_prompt': result_context})
                print('\nLLM Response after execution:', response.content, '\n')
                if 'DONE' in response.content:
                    break
            except Exception as e:
                print(f'\nError during code execution: {e}')
        else:
            print('No code detected, continuing dialogue.\n')
        ct+=1

if __name__ == "__main__":
    interactive_code_execution()    

You:  decorate a room with multiple objects. each object should have random x,y coordinates between 0,0 and 100,100. Find the 2 objects that are closest to each other. make 10 objects



LLM Response:  ```python
import random
import math

# Function to generate a random point within the given bounds
def generate_random_point():
    return (random.randint(0, 100), random.randint(0, 100))

# List to store all objects
objects = []

# Generate and add 10 objects with random coordinates
for _ in range(10):
    obj = {'x': generate_random_point()[0], 'y': generate_random_point()[1]}
    objects.append(obj)

# Function to calculate the distance between two points
def calculate_distance(point1, point2):
    return math.sqrt((point1['x'] - point2['x'])**2 + (point1['y'] - point2['y'])**2)

# Initialize minimum distance and closest pair of objects
min_distance = float('inf')
closest_pair = None

# Iterate over all pairs of objects to find the closest pair
for i in range(len(objects)):
    for j in range(i+1, len(objects)):
        dist = calculate_distance(objects[i], objects[j])
        if dist < min_distance:
            min_distance = dist
            closest_pair = (objects

You:  ok continue



LLM Response:  I'm ready to assist. What's the first prompt or question? 

No code detected, continuing dialogue.



# Adding Message Persistence

Changing to chatbot functions

In [40]:
import yaml

def load_yaml_config(file_path):
    with open(file_path, 'r') as file:
        config = yaml.safe_load(file)
    return config

llm_config_path = "llm_config.yaml"
llm_config = load_yaml_config(llm_config_path)

default_system_prompt = """
You are an AI assistant with a Python interpreter. When quantitative analysis or code execution is required for assisting your reasoning, you may generate Python code. 
Generated code should always include necessary imports and provide output relevant to the task.
If code is generated, make sure it begins with ```python and it ends using ```.
If there’s an error during execution, I will return the error message for you to adjust the code and try again.
After you generate code, I will run it and input it into the next query I submit to you. At which point, 
if you are satisfied with your answer after, and only after, you have received the code execution results, output "DONE: ANSWER" and provide the answer from the code execution.
"""

system_prompt = llm_config.get('system_prompt', default_system_prompt)
model = llm_config.get('model', 'llama3.2')
temperature = llm_config.get('temperature')

In [41]:
import os
import traceback
from io import StringIO
from contextlib import redirect_stdout

import langchain
from langchain_ollama import ChatOllama
from langchain.chains import LLMChain
from langchain.tools import Tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage



# Initialize LLM
model = ChatOllama(
    model=model,
    temperature=temperature # ranges from 0 to 1
)


In [42]:
# Define the Python execution tool (Langchain tool)
def execute_python_code(code:str)->str:
    '''
    Executes the python code provided in a string. Captures output and errors and to return them
    '''
    try:
        # Capture standard output
        f = StringIO()
        with redirect_stdout(f):
            exec(code,{})

        # Return captured output
        return f.getvalue() if f.getvalue() else "Code executed without output."
    except:
        return f'Execution error: {traceback.format_exc()}'

# Define the langchain tool for executing python code
python_executor = Tool(name='PythonExecutor',
                      func=execute_python_code,
                      description='Executes Python code and returns the output or error.'
                      )

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system", 
            f"{system_prompt}",
        ),
        MessagesPlaceholder(variable_name='messages'),
    ]
)

In [43]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, MessagesState, StateGraph

# Define a new graph
workflow = StateGraph(state_schema=MessagesState)

# Define the function that calls the model
def call_model(state: MessagesState):
    chain = prompt | model
    response = chain.invoke(state)
    return {"messages": response}

# Define the (single) node in the graph
workflow.add_edge(START, "model")
workflow.add_node("model",call_model)

# Add memory
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

Below config must be passed into the runnable every time (stores the conversation)

In [44]:
config = {"configurable": {"thread_id": "abc125"}}

## Initial Loop
Has the problem of requiring user input after running a python code given by the LLM that did not work

In [9]:
while True:
        query = input('You: ')
        # Exit if user types 'exit'
        if query.lower() == 'exit':
            print('Exiting Dialogue.')
            break
        input_messages = [HumanMessage(query)]
        response = app.invoke({'messages': input_messages},config)
        print("\n\nResponse:",response['messages'][-1].content,'\n')

        # Check for Python code blocks in the LLM response
        if "```python" in response['messages'][-1].content:
            code_snippet = response['messages'][-1].content.split("```python")[1].split("```")[0].strip()
            print('\nExecuting code snippet:\n', code_snippet)
            try:
                # Execute the code
                execution_result = python_executor.run(code_snippet)
    
                # Append execution results to continue the conversation
                print('\nExecution Result:', execution_result)
                result_context = (#f"User asked: '{user_input}'\n"
                        #f"Your response: '{response.content}'\n"
                        f"Your Code Execution Result: {execution_result}\n"   
                )
                
                # Pass the result back as context to refine or complete the response
                response = app.invoke({'messages': result_context},config)
                print('\nLLM Response after execution:', response['messages'][-1].content, '\n')
                if 'DONE' in response['messages'][-1].content:
                    break
            except Exception as e:
                print(f'\nError during code execution: {e}')
        else:
            print('No code detected, continuing dialogue.\n')

You:  hardcode the XY positions of 10 random objects. Then find the two objects that are closest together




Response: ```python
import math
import random

# Hardcode the XY positions of 10 random objects
objects = [(random.uniform(0, 100), random.uniform(0, 100)) for _ in range(10)]

# Function to calculate distance between two points
def distance(point1, point2):
    return math.sqrt((point1[0] - point2[0])**2 + (point1[1] - point2[1])**2)

# Find the minimum distance and the corresponding objects
min_distance = float('inf')
closest_objects = None

for i in range(len(objects)):
    for j in range(i+1, len(objects)):
        dist = distance(objects[i], objects[j])
        if dist < min_distance:
            min_distance = dist
            closest_objects = (i, j)

# Print the results
print("XY positions of 10 random objects:")
for i, obj in enumerate(objects):
    print(f"Object {i+1}: ({obj[0]}, {obj[1]})")

print("\nThe two objects that are closest together are:")
print(f"Objects {closest_objects[0]+1} and {closest_objects[1]+1}")
print(f"Their distance is: {min_distance:.2f}")

# Print 

## Improved Loop - catches and loops if python error occurs from LLM-generated code

In [49]:
retries = 0
max_retries = 3

while True:
        query = input('You: ')
        # Exit if user types 'exit'
        if query.lower() == 'exit':
            print('Exiting Dialogue.')
            break
            
        input_messages = [HumanMessage(query)]
        response = app.invoke({'messages': input_messages},config)
        print("\n\nResponse:",response['messages'][-1].content,'\n')

        # Check for Python code blocks in the LLM response
        if "```python" in response['messages'][-1].content:
            code_snippet = response['messages'][-1].content.split("```python")[1].split("```")[0].strip()

            while retries < max_retries:
                print('\nExecuting code snippet:\n', code_snippet)
                try:
                    # Execute the code
                    execution_result = python_executor.run(code_snippet)
        
                    # Append execution results to continue the conversation
                    print('\nExecution Result:', execution_result)
                    result_context = (f"Your Code Execution Result: {execution_result}\n")
                    
                    # Pass the result back as context to refine or complete the response
                    response = app.invoke({'messages': [HumanMessage(result_context)]}, config)
                    print('\nLLM Response after execution:', response['messages'][-1].content, '\n')

                    # Update the code snippet if the LLM provided refined code
                    if "```python" in response['messages'][-1].content:
                        code_snippet = response['messages'][-1].content.split("```python")[1].split("```")[0].strip()
                    else:
                        print('No refined code provided, exiting refinement loop.')
                        break
                        
                except Exception as e:
                    retries += 1
                    # Provide erro to LLM for feedback
                    print(f'\nError during code execution (Attempt {retries}/{max_retries}): {e}')
                    error_context = f"Error during code execution: {str(e)}"
                    response = app.invoke({'messages': [HumanMessage(error_context)]},config)
                    print('\nLLM Response after error: ', response['messages'][-1].content, '\n')
                    
                    # Update the code snippet if the LLM provided refined code
                    if "```python" in response['messages'][-1].content:
                        code_snippet = response['messages'][-1].content.split("```python")[1].split("```")[0].strip()
                    else:
                        print('No refined code provided, exiting refinement loop.')
                        break
                        
            if 'DONE' in response['messages'][-1].content:
                break
            if retries == max_retries:
                print('Max retries reached, exiting code execution loop.')    
        else:
            print('No code detected, continuing dialogue.\n')

You:  hardcode 10 random household objects and their XY positions. Do not use random functions. Then, find the two objects that are closest together




Response: ```python
# Hardcoded XY positions of 10 household objects
objects = {
    'Chair': (1, 2),
    'Table': (4, 6),
    'Lamp': (7, 3),
    'Bookshelf': (0, 5),
    'TV': (3, 8),
    'Refrigerator': (9, 1),
    'Sofa': (2, 4),
    'Desk': (6, 7),
    'Bed': (5, 2),
    'Washing Machine': (8, 6)
}

# Function to calculate distance between two points
def distance(x1, y1, x2, y2):
    return ((x2 - x1)**2 + (y2 - y1)**2) ** 0.5

# Initialize minimum distance and closest objects
min_distance = float('inf')
closest_objects = None

# Iterate over all pairs of objects
for obj1 in objects:
    for obj2 in objects:
        if obj1 != obj2:  # Don't compare an object with itself
            x1, y1 = objects[obj1]
            x2, y2 = objects[obj2]
            dist = distance(x1, y1, x2, y2)
            if dist < min_distance:
                min_distance = dist
                closest_objects = (obj1, obj2)

# Print the two objects that are closest together
print(f"The two objects that 

hardcode 10 random household objects and their XY positions. Do not use random functions. Then, find the two objects that are closest together

In [2]:
from llm_script import LLM_Solver
import os
import traceback
from io import StringIO
from contextlib import redirect_stdout

import langchain
from langchain_ollama import ChatOllama
from langchain.chains import LLMChain
from langchain.tools import Tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

ps = LLM_Solver()
string_data = '''
InstanceID class X_location Y_location
1 class:Chair x:0.83293 y:0.94212
2 class:Chair x:0.32390 y:0.83282
3 class:Chair x:0.56789 y:0.12345
4 class:Door x:0.10000 y:0.90000
5 class:Table x:0.11111 y:0.11111
6 class:Lamp x:0.22222 y:0.22222
7 class:Book x:0.33333 y:0.33333
8 class:Plant x:0.44444 y:0.44444
9 class:Cup x:0.55555 y:0.55555
10 class:Shelf x:0.66666 y:0.66666
11 class:Notebook x:0.77777 y:0.77777
12 class:Pen x:0.88888 y:0.88888
13 class:Phone x:0.99999 y:0.99999
14 class:Keyboard x:0.12121 y:0.23232
15 class:Mouse x:0.34343 y:0.45454
16 class:Monitor x:0.56565 y:0.67676
17 class:Speaker x:0.78787 y:0.89898
18 class:Bag x:0.90909 y:0.10101
19 class:Bottle x:0.21212 y:0.32323
20 class:Clock x:0.43434 y:0.54545
'''
ps.run(string_data)

Using thread_id: 1234
You are an AI assistant with a Python interpreter. When quantitative analysis or code execution is required for assisting your reasoning, you may generate Python code. 
Generated code should always include necessary imports and provide output relevant to the task.
If code is generated, make sure it begins with ```python and it ends using ```.
After you generate code, your Python execution tool will run it and provide it to you in the next AIMessage. 
If there is an error during execution, I will return the error message for you to adjust the code and try again.
At which point, if you believe your answer is correct after, and only after, you have received the code execution results, output "DONE: ANSWER" and provide the answer from the code execution.
Do not state "DONE: ANSWER" until after you recieve the code execution results.
You must wait for the code results to be provided to you, before you give an answer.
  Your objective is to solve a 3D Referring Expressi

# Other items to explore
Human in the loop Tool: https://python.langchain.com/docs/how_to/tools_human/

Chatbot vs agent